In [ ]:
import os, gc, json, math, time, warnings
import numpy as np, pandas as pd, torch, timm, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from tqdm import tqdm
warnings.filterwarnings("ignore"); dev = "cuda"

OUT = r"G:\runs"; VW = os.path.join(OUT, "vinbig"); VT = os.path.join(OUT, "convnext_dg"); os.makedirs(VT, exist_ok=True)
FIG = os.path.join(VT, "figures"); os.makedirs(FIG, exist_ok=True)
ARCH = "convnext_tiny.fb_in1k"
EPOCHS, SEEDS, BS, LR, WD = 7, [0, 1, 2], 64, 1e-4, 1e-4          # identical to your DenseNet / ConvNeXt runs
LABELS6 = ["Cardiomegaly", "Edema", "Consolidation", "Atelectasis", "Pneumothorax", "Pleural Effusion"]
FIND5 = ["Cardiomegaly", "Consolidation", "Atelectasis", "Pneumothorax", "Pleural Effusion"]; L = 6
SITE = {"A": "NIH", "B": "CheXpert", "C": "VinBigData"}
plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Times", "DejaVu Serif"], "mathtext.fontset": "stix", "font.size": 8,
    "axes.labelsize": 8, "axes.titlesize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7, "axes.linewidth": 0.6, "figure.dpi": 150, "pdf.fonttype": 42})
COL1, COL2 = 3.5, 7.16
def save_fig(fig, name):
    fig.savefig(os.path.join(FIG, name + ".pdf"), bbox_inches="tight"); fig.savefig(os.path.join(FIG, name + ".png"), dpi=600, bbox_inches="tight"); plt.show()
print(torch.cuda.get_device_name(0), "| output ->", VT)

In [ ]:
nih = pd.read_csv(r"G:\nih_split.csv"); off = nih.official.values == "test"; te_i = np.where(off)[0]
tr_i = np.where(nih.split_rev.values == "train")[0]; va_i = np.where(nih.split_rev.values == "val")[0]
NIH_A = np.memmap(r"G:\nih_224.u8", dtype=np.uint8, mode="r", shape=(len(nih), 224, 224)); Yn = nih[LABELS6].values.astype(int)

cx = pd.read_csv(r"G:\chexpert_final_clean.csv"); cxt = cx[cx.split == "test"].reset_index(drop=True)
cxm = cx[cx.split.isin(["train", "val"])].reset_index(drop=True)
cx_tr = np.where(cxm.split.values == "train")[0]; cx_va = np.where(cxm.split.values == "val")[0]
CX_A = np.memmap(r"G:\cx_trainval_224.u8", dtype=np.uint8, mode="r", shape=(len(cxm), 224, 224)); CX_T = np.load(r"G:\cx_test_224.npy")
Ycm = cxm[LABELS6].values.astype(int); Yc = cxt[LABELS6].values.astype(int)

vb = pd.read_csv(os.path.join(VW, "vinbig_image_labels.csv")); NVB = len(vb)
VB_A = np.memmap(os.path.join(VW, "vb_train_224.u8"), dtype=np.uint8, mode="r", shape=(NVB, 224, 224))
Yvb = np.zeros((NVB, L), int)
for f in FIND5: Yvb[:, LABELS6.index(f)] = vb[f].values
VALID_C = np.array([l != "Edema" for l in LABELS6])                    # VinBigData has no Edema -> masked in the loss and in the metrics
sp = os.path.join(OUT, "fair_ens", "vinbig_split.csv")                  # same C split as full11
if os.path.exists(sp):
    s_ = pd.read_csv(sp); vb_va = np.where(s_.split.values == "val")[0]; vb_tr = np.where(s_.split.values == "train")[0]
else:
    perm = np.random.default_rng(0).permutation(NVB); vb_va = np.sort(perm[:NVB // 5]); vb_tr = np.sort(perm[NVB // 5:])

SRC = {"A": dict(arr=NIH_A, Y=Yn,  tr=tr_i,  va=va_i,  valid=np.ones(L, bool)),
       "B": dict(arr=CX_A,  Y=Ycm, tr=cx_tr, va=cx_va, valid=np.ones(L, bool)),
       "C": dict(arr=VB_A,  Y=Yvb, tr=vb_tr, va=vb_va, valid=VALID_C)}
TEST = {"A": (NIH_A, te_i), "B": (CX_T, np.arange(len(CX_T))), "C": (VB_A, np.arange(NVB))}   # C is a test site only when it is not a training source
YT = {"A": Yn[te_i], "B": Yc, "C": Yvb}; VALID = {"A": np.ones(L, bool), "B": np.ones(L, bool), "C": VALID_C}
print({k: (len(v["tr"]), len(v["va"])) for k, v in SRC.items()}, "| test:", len(te_i), len(CX_T), NVB)

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
def make_model(pretrained=True): return timm.create_model(ARCH, pretrained=pretrained, num_classes=L).to(dev)     # no drop-path, like your earlier ConvNeXt runs
def gpu_batch(A, idx, flip=False):
    x = torch.from_numpy(np.ascontiguousarray(A[np.asarray(idx)])).to(dev).float().div(255).unsqueeze(1)
    if flip:
        f = torch.rand(len(idx), device=dev) < 0.5; x = torch.where(f.view(-1, 1, 1, 1), x.flip(-1), x)
    return (x.repeat(1, 3, 1, 1) - MEAN) / STD
@torch.inference_mode()
def predict(m, A, idx, bs=128):
    m.eval(); out = []
    for i in range(0, len(idx), bs):
        with torch.autocast("cuda"): o = m(gpu_batch(A, idx[i:i + bs]))
        out.append(torch.sigmoid(o.float()).cpu())
    return torch.cat(out).numpy()
def fauc(y, s):
    y = np.asarray(y).astype(bool); r = rankdata(s); n1 = y.sum(); n0 = len(y) - n1
    return (r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
def lab_auc(Y, P, valid): return np.array([fauc(Y[:, j], P[:, j]) if valid[j] and 5 <= Y[:, j].sum() <= len(Y) - 5 else np.nan for j in range(L)])
def mauc(Y, P, valid): return float(np.nanmean(lab_auc(Y, P, valid)))
def val_scores(m, srcs):                                    # SOURCE validation only, never the target
    return {k: mauc(SRC[k]["Y"][SRC[k]["va"]], predict(m, SRC[k]["arr"], SRC[k]["va"]), SRC[k]["valid"]) for k in srcs}

def batches_bal(srcs, seed, ep):                            # equal number of images from each hospital in every step
    rng = np.random.default_rng(seed * 100 + ep); h = BS // len(srcs)
    for _ in range(math.ceil(sum(len(SRC[k]["tr"]) for k in srcs) / BS)):
        yield {k: np.sort(rng.choice(SRC[k]["tr"], h, replace=False)) for k in srcs}
class GRL(torch.autograd.Function):                         # gradient reversal for DANN
    @staticmethod
    def forward(ctx, x, lam): ctx.lam = lam; return x.view_as(x)
    @staticmethod
    def backward(ctx, g): return -ctx.lam * g, None
def coral_loss(fa, fb):                                     # match feature covariances of the two hospitals
    d = fa.size(1); return ((torch.cov(fa.T) - torch.cov(fb.T)) ** 2).sum() / (4 * d * d)

METHS = ["ERM (balanced)", "DANN", "CORAL", "GroupDRO"]; TAG = {"ERM (balanced)": "erm_bal", "DANN": "dann", "CORAL": "coral", "GroupDRO": "gdro"}
POOLED = ["A+B", "A+C", "B+C"]; UNSEEN = {"A+B": "C", "A+C": "B", "B+C": "A"}
def base_path(ts, s): return os.path.join(VT, f"cnx_{ts.replace('+', 'p')}_seed{s}")
def mpath(ts, meth, s): return base_path(ts, s) + "_" + TAG[meth]

In [ ]:
def train_dg_cnx(ts, meth, seed, dann_w=0.1, coral_w=1.0, dro_eta=0.01):       # defaults = your DenseNet runs (DANN_W, CORAL_W, DRO_ETA)
    srcs = ts.split("+"); b = mpath(ts, meth, seed); BEST, LAST, HIST, PRD = b + ".pt", b + "_last.pt", b + "_history.csv", b + "_preds.npz"
    if os.path.exists(PRD): print("skip (finished):", os.path.basename(b)); return
    torch.manual_seed(seed); np.random.seed(seed)
    m = make_model(); disc = nn.Sequential(nn.Linear(m.num_features, 256), nn.ReLU(), nn.Linear(256, len(srcs))).to(dev) if meth == "DANN" else None
    opt = torch.optim.AdamW(list(m.parameters()) + (list(disc.parameters()) if disc else []), lr=LR, weight_decay=WD)
    spe = math.ceil(sum(len(SRC[k]["tr"]) for k in srcs) / BS); sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS * spe)
    scaler = torch.amp.GradScaler("cuda"); hist, best, start = [], 0.0, 0; q = torch.ones(len(srcs), device=dev) / len(srcs)
    if os.path.exists(LAST):
        ck = torch.load(LAST, map_location=dev, weights_only=False)
        m.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sch.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"]); q = ck["q"].to(dev)
        if disc: disc.load_state_dict(ck["disc"])
        hist, best, start = ck["hist"], ck["best"], ck["ep"] + 1; print("resumed, next epoch", start + 1)
    tt = lambda a, dt: torch.from_numpy(np.concatenate(a)).to(dev).to(dt)
    for ep in range(start, EPOCHS):
        m.train(); t0 = time.time(); tl = torch.zeros((), device=dev)
        for k, batch in enumerate(tqdm(batches_bal(srcs, seed, ep), total=spe, leave=False, desc=f"ConvNeXt {ts} {meth} seed {seed} ep {ep + 1}/{EPOCHS}")):
            xs, ys, ms, ds = [], [], [], []
            for i, s_ in enumerate(srcs):
                idx = batch[s_]; xs.append(gpu_batch(SRC[s_]["arr"], idx, flip=True)); ys.append(SRC[s_]["Y"][idx])
                ms.append(np.tile(SRC[s_]["valid"], (len(idx), 1))); ds.append(np.full(len(idx), i))
            x = torch.cat(xs); y, mk, d = tt(ys, torch.float32), tt(ms, torch.float32), tt(ds, torch.long)
            with torch.autocast("cuda"):
                f = m.forward_head(m.forward_features(x), pre_logits=True); z = m.get_classifier()(f)
            f = f.float(); ls = (F.binary_cross_entropy_with_logits(z.float(), y, reduction="none") * mk).sum(1) / mk.sum(1).clamp(min=1)   # per-sample loss, Edema masked for C
            if meth == "ERM (balanced)": loss = ls.mean()
            elif meth == "GroupDRO":
                gl = torch.stack([ls[d == i].mean() for i in range(len(srcs))])
                q = q * torch.exp(dro_eta * gl.detach()); q = q / q.sum(); loss = (q * gl).sum()
            elif meth == "CORAL": loss = ls.mean() + coral_w * coral_loss(f[d == 0], f[d == 1])
            else:                                                                           # DANN: 0.1 x domain loss, lambda ramps 0 -> 1
                p = (ep * spe + k) / (EPOCHS * spe); lam = 2 / (1 + math.exp(-10 * p)) - 1
                loss = ls.mean() + dann_w * F.cross_entropy(disc(GRL.apply(f, lam)), d)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step(); tl += loss.detach()
        vs = val_scores(m, srcs); vm = float(np.mean(list(vs.values())))                    # same source-only, mean-over-hospitals selection
        row = dict(epoch=ep + 1, train_loss=float(tl) / spe, val_mean=vm, **{f"val_{k}": v for k, v in vs.items()}, sec=time.time() - t0)
        hist.append(row); pd.DataFrame(hist).to_csv(HIST, index=False)
        print(f"ConvNeXt {ts} {meth} seed {seed} ep {ep + 1} | loss {row['train_loss']:.4f} | val {vs} | {row['sec'] / 60:.1f} min" + (f" | q={q.cpu().numpy().round(2)}" if meth == "GroupDRO" else ""))
        if vm > best: best = vm; torch.save(m.state_dict(), BEST)
        torch.save(dict(model=m.state_dict(), opt=opt.state_dict(), sched=sch.state_dict(), scaler=scaler.state_dict(), hist=hist, best=best, ep=ep, q=q.cpu(), **({"disc": disc.state_dict()} if disc else {})), LAST)
    m.load_state_dict(torch.load(BEST, map_location=dev, weights_only=True)); z = {}
    z["A_test_P"] = predict(m, NIH_A, te_i); z["B_test_P"] = predict(m, CX_T, np.arange(len(CX_T)))
    if "C" not in srcs: z["C_test_P"] = predict(m, VB_A, np.arange(NVB))                    # C is a clean test site only if it was not trained on
    for k in ["A", "B"] + (["C"] if "C" in srcs else []): z[f"{k}_val_P"] = predict(m, SRC[k]["arr"], SRC[k]["va"])
    np.savez(PRD, **z); del m, opt, sch, scaler, disc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
RUN_ORDER = ["A+B", "A+C", "B+C"]; SEEDS_RUN = SEEDS; STOP_AFTER = None          # first test: SEEDS_RUN = [0] and STOP_AFTER = 2
PLAN = [(ts, me, s) for ts in RUN_ORDER for me in METHS for s in SEEDS_RUN]
done = lambda ts, me, s: os.path.exists(mpath(ts, me, s) + "_preds.npz")
display(pd.DataFrame({ts: {me: f"{sum(done(ts, me, s) for s in SEEDS)}/{len(SEEDS)}" for me in METHS} for ts in RUN_ORDER}))
for ts, me, s in [p for p in PLAN if not done(*p)][:STOP_AFTER]:
    try: train_dg_cnx(ts, me, s)
    except torch.cuda.OutOfMemoryError:
        print("OOM: set BS = 32 in Cell C0, restart the kernel, rerun C0-C3, then rerun this cell (it resumes)."); break
print("all finished" if not [p for p in PLAN if not done(*p)] else "runs left: rerun this cell")

In [ ]:
FE = os.path.join(OUT, "fair_ens"); sig = lambda z: 1 / (1 + np.exp(-z))
def natural(ts, s):
    f = os.path.join(FE, f"logits_ConvNeXt_{ts.replace('+', 'p')}_erm_nat_seed{s}.npz"); return sig(np.load(f)[UNSEEN[ts]]) if os.path.exists(f) else None
rows = []
for ts in POOLED:
    T = UNSEEN[ts]
    for me in ["ERM (natural)"] + METHS:
        v = []
        for s in SEEDS:
            if me == "ERM (natural)": P = natural(ts, s)
            else:
                f = mpath(ts, me, s) + "_preds.npz"; P = np.load(f)[f"{T}_test_P"] if os.path.exists(f) else None
            if P is not None: v.append(mauc(YT[T], P, VALID[T]))
        if v: rows.append(dict(train=ts, unseen=T, method=me, seeds=len(v), auroc=np.mean(v), sd=np.std(v, ddof=1) if len(v) > 1 else 0))
R = pd.DataFrame(rows); R.to_csv(os.path.join(VT, "table_convnext_dg.csv"), index=False)
ALL = ["ERM (natural)"] + METHS
display(R.assign(v=R.apply(lambda r: f"{r.auroc:.3f} ± {r.sd:.3f} (n={r.seeds})", axis=1)).pivot(index="method", columns="train", values="v").reindex(ALL))

fig, ax = plt.subplots(figsize=(COL2, 2.9)); w = .16; cols = ["0.12", "0.38", "0.58", "0.75", "0.9"]; hat = ["", "///", "\\\\\\", "xxx", "..."]
for i, me in enumerate(ALL):
    g = R[R.method == me].set_index("train").reindex(POOLED)
    ax.bar(np.arange(3) + (i - 2) * w, g.auroc, w, yerr=g.sd, capsize=1.5, color=cols[i], hatch=hat[i], edgecolor="k", lw=.4, label=me)
ax.set_xticks(range(3)); ax.set_xticklabels([f"{t}→{UNSEEN[t]}\n({SITE[UNSEEN[t]]})" for t in POOLED]); ax.set_ylim(.6, None); ax.set(ylabel="Macro AUROC (unseen site)", title="ConvNeXt-T: DG methods vs ERM"); ax.legend(frameon=False, fontsize=6, ncol=2); ax.grid(True, axis="y", alpha=.3)
fig.tight_layout(); save_fig(fig, "fig_convnext_dg_methods")